# Phase 2: Exploratory Data Analysis

**BOL-LPP Reproduction Project**

This notebook explores the raw datasets downloaded in Phase 1 before any preprocessing, to inform decisions made in Phase 3.

**Goals:**
- Characterise the distribution of ERCOT day-ahead prices, including negative prices and scarcity spikes
- Examine temporal patterns in load and price (hour-of-day, day-of-week, seasonal)
- Assess the load–price relationship and correlations between candidate input features
- Identify missing values and gaps in each dataset
- Verify that the price, load, and weather series align on a common hourly timestamp index

**Inputs:** `data/raw/ercot_dam_spp_zones_2013_2018.csv`, `data/raw/ercot_hourly_load_zones_2013_2018.csv`, `data/raw/weather_north_2013_2018.csv`

---
**Author:** Jahandad Irfan

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW_DIR = Path("../data/raw")

prices = pd.read_csv(RAW_DIR / "ercot_dam_spp_zones_2013_2018.csv")
load = pd.read_csv(RAW_DIR / "ercot_hourly_load_zones_2013_2018.csv")
weather = pd.read_csv(RAW_DIR / "weather_north_2013_2018.csv")

print("PRICES:", prices.shape)
print(prices.dtypes)
print(prices.head(3))
print("\nLOAD:", load.shape)
print(load.dtypes)
print(load.head(3))
print("\nWEATHER:", weather.shape)
print(weather.dtypes)
print(weather.head(3))

Matplotlib is building the font cache; this may take a moment.


PRICES: (157752, 8)
Time               object
Interval Start     object
Interval End       object
Location           object
Location Type      object
Market             object
SPP               float64
Zone               object
dtype: object
                        Time             Interval Start  \
0  2013-01-01 00:00:00-06:00  2013-01-01 00:00:00-06:00   
1  2013-01-01 00:00:00-06:00  2013-01-01 00:00:00-06:00   
2  2013-01-01 00:00:00-06:00  2013-01-01 00:00:00-06:00   

                Interval End    Location Location Type            Market  \
0  2013-01-01 01:00:00-06:00    LZ_NORTH     Load Zone  DAY_AHEAD_HOURLY   
1  2013-01-01 01:00:00-06:00  LZ_HOUSTON     Load Zone  DAY_AHEAD_HOURLY   
2  2013-01-01 01:00:00-06:00    LZ_SOUTH     Load Zone  DAY_AHEAD_HOURLY   

     SPP   Zone  
0  23.12  North  
1  23.09  Coast  
2  23.09  South  

LOAD: (157752, 4)
Interval Start     object
Interval End       object
Zone               object
Load_MW           float64
dtype: object
       

In [2]:
load_all_check = pd.read_csv(RAW_DIR / "ercot_hourly_load_zones_2013_2018.csv")
print(load_all_check.groupby("Zone")["Load_MW"].describe())

         count          mean          std          min          25%  \
Zone                                                                  
Coast  52583.0  11498.074572  2605.482810  6602.878600  9621.753046   
North  52583.0    838.262200   176.104025   522.306095   714.309402   
South  52583.0   3322.099543   824.131500  1665.982938  2708.097051   

                50%           75%           max  
Zone                                             
Coast  10873.786330  12906.846477  20269.849015  
North    794.614491    931.604563   1521.845309  
South   3157.162653   3849.347735   6176.106417  


In [3]:
import gridstatus
ercot = gridstatus.Ercot()

sample = ercot.get_hourly_load_post_settlements(date="2016-07-01", end="2016-07-07")
zone_cols = ["Coast", "East", "Far West", "North", "North Central",
             "South", "South Central", "West", "ERCOT"]
print(sample[zone_cols].mean().round(0))

2026-09-04 21:29:10 - INFO - Fetching historical load data for year 2016
2026-09-04 21:29:11 - DEBUG - Changing timezone for DST duplicate at 2016-11-06 01:00:00-06:00


Coast            11694.0
East              1398.0
Far West          2143.0
North              823.0
North Central    12953.0
South             3395.0
South Central     6484.0
West              1115.0
ERCOT            40006.0
dtype: float64
